<a href="https://colab.research.google.com/github/LindsayRendon/campaigns-demo/blob/master/04_DRL_PPO_CartPole_StableBaselines3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Entrenamiento de un agente con PPO en CartPole

## 2. Objetivo
Entrenar un agente de Inteligencia Artificial utilizando Aprendizaje por Refuerzo Profundo (Deep Reinforcement Learning) para que aprenda a mantener un palo en equilibrio sobre un carro móvil.

## 3. Qué problema resuelve
En el mundo real, los sistemas robóticos, los coches autónomos o los sistemas de control industrial (ej. controlar la temperatura de un alto horno) no siempre tienen millones de ejemplos etiquetados sobre "qué hacer" en cada milisegundo. Este modelo resuelve el problema de la **toma de decisiones secuenciales**, permitiendo que la máquina descubra la estrategia óptima por sí misma interactuando con su entorno.

## 4. Dataset (Entorno) usado
No hay un dataset tradicional (no hay archivos CSV ni carpetas de imágenes). Usamos un **Entorno de Gymnasium** llamado `CartPole-v1`. Las reglas son simples: tienes un carro que solo puede moverse a la izquierda o a la derecha, y un palo articulado encima. El objetivo es mover el carro para que el palo no se caiga. Si el palo se inclina demasiado o el carro se sale de la pantalla, el juego termina.

---
### Conceptos teóricos clave aplicados en este Notebook:



* **Agente:** Es el "cerebro" o protagonista que toma las decisiones. En este caso, es nuestra red neuronal PPO que decide cómo mover el carro.
* **Entorno:** Es el universo físico (o simulado) donde interactúa el agente. Aquí, el entorno incluye la física del carro, la gravedad, el palo y las reglas del juego.
* **Acción:** Son los movimientos que puede hacer el agente. En CartPole solo hay dos: mover el carro a la Izquierda (0) o a la Derecha (1).
* **Recompensa (Reward):** Es el puntaje. En CartPole, el entorno te da +1 de recompensa por cada paso de tiempo que logres mantener el palo en pie. El objetivo del agente es maximizar esa recompensa (llegar a 500 puntos).
* **Política (Policy):** Es el manual de instrucciones mental del agente. Es una función que relaciona lo que el agente ve (el estado: posición del carro, ángulo del palo) con la acción que debe tomar (izquierda o derecha). El entrenamiento busca encontrar la política óptima.
* **PPO (Proximal Policy Optimization):** Es uno de los algoritmos de aprendizaje por refuerzo más potentes y estables. Es el mismo algoritmo que usa OpenAI para entrenar y alinear a ChatGPT (RLHF). Funciona ajustando la política de forma "cercana" (proximal), evitando dar saltos bruscos que arruinen lo aprendido.
* **Supervisado vs Por Refuerzo:** En el Supervisado hay un humano que da la "respuesta correcta" (ej. "Esta foto es un gato"). En el Por Refuerzo nadie le dice al carro cómo moverse; el carro prueba cosas al azar y la señal de aprendizaje es solo la recompensa (+1 o "Fin del juego").

In [1]:
# Instalación de librerías para Reinforcement Learning y simulación de pantalla en Colab
# Como vamos a grabar un vídeo de un juego dentro de Colab (que no tiene pantalla física), necesitamos instalar unas librerías especiales para crear una "pantalla virtual".
!apt-get update && apt-get install -y xvfb
!pip install stable-baselines3[extra] gymnasium pyvirtualdisplay moviepy

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,303 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB

In [2]:
# 5. Librerías (Vamos a importar las librerías y encender la "pantalla virtual")
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from gymnasium.wrappers import RecordVideo
import pyvirtualdisplay
import base64
from IPython.display import HTML
import os

# Encender pantalla virtual oculta para Colab
display = pyvirtualdisplay.Display(visible=0, size=(1400, 900))
display.start()

print("Librerías cargadas y pantalla virtual iniciada.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Librerías cargadas y pantalla virtual iniciada.


In [3]:
# 6. Creación del entorno y evaluación ANTES de entrenar
# Vamos a crear el entorno y a meter a un agente "recién nacido" (sin entrenar) para ver qué tan mal lo hace
env_id = "CartPole-v1"
env = gym.make(env_id)

# Creamos el modelo PPO (política MlpPolicy indica que usará una red neuronal densa clásica)
model = PPO("MlpPolicy", env, verbose=0)

# Evaluamos al modelo antes de entrenarlo (haremos 10 partidas de prueba)
mean_reward_before, std_reward_before = evaluate_policy(model, env, n_eval_episodes=10)

print(f"Evaluación ANTES del entrenamiento:")
print(f"Recompensa media: {mean_reward_before:.2f} (Máximo posible: 500)")
print("El agente aguanta muy pocos pasos antes de que se le caiga el palo.")

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Evaluación ANTES del entrenamiento:
Recompensa media: 105.00 (Máximo posible: 500)
El agente aguanta muy pocos pasos antes de que se le caiga el palo.


In [4]:
# 7. Entrenamiento del Modelo
#¡Hora de entrenar! Vamos a dejar que el agente juegue durante 20,000 pasos de tiempo.
print("Iniciando entrenamiento del agente (prueba y error)...")

# Entrenamos por 20,000 ciclos. (En tu ordenador esto tomará unos 10-30 segundos).
model.learn(total_timesteps=20000)

print("¡Entrenamiento finalizado!")

Iniciando entrenamiento del agente (prueba y error)...
¡Entrenamiento finalizado!


In [5]:
# 8. Evaluación DESPUÉS del entrenamiento
# Comprobamos la recompensa DESPUÉS del entrenamiento
mean_reward_after, std_reward_after = evaluate_policy(model, env, n_eval_episodes=10)

print(f"Evaluación DESPUÉS del entrenamiento:")
print(f"Recompensa media: {mean_reward_after:.2f} (Máximo posible: 500)")
print("¡El agente debería haber alcanzado los 500 puntos o estar muy cerca!")

Evaluación DESPUÉS del entrenamiento:
Recompensa media: 500.00 (Máximo posible: 500)
¡El agente debería haber alcanzado los 500 puntos o estar muy cerca!


In [6]:
# 9. Grabación de vídeo del agente jugando
# Vamos a grabar al agente jugando para poder verlo en Colab.
# Definimos la carpeta donde se guardará el vídeo
video_folder = "./videos"
os.makedirs(video_folder, exist_ok=True)

# Creamos un entorno especial que graba lo que pasa
env_record = gym.make(env_id, render_mode="rgb_array")
env_record = RecordVideo(env_record, video_folder=video_folder, name_prefix="cartpole_ppo", disable_logger=True)

# Ejecutamos 1 partida completa para grabarla
obs, info = env_record.reset()
done = False
truncated = False

while not (done or truncated):
    # El agente observa el entorno y decide la mejor acción
    action, _states = model.predict(obs, deterministic=True)
    # Ejecutamos la acción en el entorno
    obs, reward, done, truncated, info = env_record.step(action)

env_record.close()
print("Vídeo grabado exitosamente.")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Vídeo grabado exitosamente.


In [7]:
# Mostramos el vídeo directamente aquí en el cuaderno.
# Función para incrustar el vídeo mp4 en el HTML del cuaderno
def mostrar_video(directory):
    html = []
    for file in os.listdir(directory):
        if file.endswith(".mp4"):
            filepath = os.path.join(directory, file)
            video_b64 = base64.b64encode(open(filepath, "rb").read()).decode('ascii')
            html.append(f'''
            <video width="600" controls autoplay loop>
                <source src="data:video/mp4;base64,{video_b64}" type="video/mp4" />
            </video>
            ''')
    return HTML("<br>".join(html))

# Visualizar el vídeo grabado
mostrar_video(video_folder)

## 10. Conclusión
El entrenamiento ha sido un éxito rotundo. Al iniciar el experimento, el agente no sabía cómo reaccionar a la gravedad y el palo se caía inmediatamente (con recompensas muy bajas, alrededor de 9 a 20 puntos). Tras 20,000 iteraciones de prueba y error utilizando el algoritmo PPO, el agente aprendió la política óptima, alcanzando una recompensa constante de 500 puntos (el límite máximo del juego), logrando un equilibrio perfecto.

## 11. Qué aprendí
* **El poder de la experiencia:** Aprendí que en el Aprendizaje por Refuerzo, la red neuronal no aprende de respuestas predefinidas, sino de las consecuencias de sus propios actos. El agente descubrió por sí mismo que moverse hacia el lado al que cae el palo contrarresta la inclinación.
* **Estabilidad del PPO:** El algoritmo Proximal Policy Optimization demostró ser sumamente rápido y eficiente. Al observar que el modelo alcanzó el éxito en tan solo unos segundos de procesamiento computacional, entendí por qué se usa en sistemas industriales críticos y en el entrenamiento de Modelos de Lenguaje Grandes (LLMs).
* **Ausencia de características manuales:** Comprobé la superioridad del Aprendizaje Profundo. No tuve que programar ninguna fórmula de física newtoniana, ni calcular ángulos ni aceleraciones. La red `MlpPolicy` recibió los números crudos del estado del juego y los mapeó directamente hacia las probabilidades de la mejor acción a tomar.